# 03 — YOLOv8 Training
Sanity check with YOLOv8n (5 epochs), then full training with YOLOv8m (50 epochs).

In [1]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "dataset").is_dir() and (_root / "src").is_dir():
        DATASET_DIR = _root / "dataset"
        break
else:
    raise FileNotFoundError("Repo root not found (need dataset/ and src/).")

# Prefer your preprocessed dataset if it exists; otherwise use dataset/
PREPROCESSED_DIR = DATASET_DIR / "bdd100k_preprocessing"
if PREPROCESSED_DIR.is_dir():
    DATA_DIR = PREPROCESSED_DIR
    print(f"Using preprocessed dataset: {DATA_DIR}")
else:
    DATA_DIR = DATASET_DIR
    print(f"Using dataset root: {DATA_DIR}")

Using dataset root: C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\dataset


In [2]:
!pip install ultralytics --no-deps

In [3]:
import sys
import os
from pathlib import Path
import numpy as np

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "dataset").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and dataset/).")

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.utils import seed_everything, log_environment

seed_everything()
log_environment()

PyTorch: 2.11.0.dev20260214+cu128
Ultralytics: 8.4.14
GPU: NVIDIA GeForce RTX 5090
CUDA: 12.8


In [4]:
import os
import shutil
import re

DATA_CONFIG = _root / "configs" / "yolov8_bdd100k.yaml"
PROJECT_DIR = _root / "outputs" / "bdd100k_project/runs"

os.makedirs(PROJECT_DIR, exist_ok=True)

In [5]:
AUGMENTATION = dict(
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    translate=0.1,
    scale=0.5,
    degrees=0.0,
)

## Sanity Check — YOLOv8n (5 epochs)

In [9]:
from ultralytics import YOLO

model_n = YOLO("yolov8n.pt")

results_n = model_n.train(
    data=DATA_CONFIG,
    epochs=5,
    imgsz=640,
    batch=16,
    name="yolov8n_bdd100k_sanity",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    **AUGMENTATION,
)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.0 torch-2.11.0.dev20260214+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\configs\yolov8_bdd100k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=t

In [11]:
print("Sanity check complete.")
print(f"Results saved to: {os.path.join(PROJECT_DIR, 'yolov8n_bdd100k_sanity')}")

import glob
sanity_dir = os.path.join(PROJECT_DIR, "yolov8n_bdd100k_sanity")
for f in sorted(glob.glob(os.path.join(sanity_dir, "*.png"))):
    print(f"  {os.path.basename(f)}")

Sanity check complete.
Results saved to: C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\runs\yolov8n_bdd100k_sanity
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  confusion_matrix.png
  confusion_matrix_normalized.png
  results.png


## Full Training — YOLOv8m (50 epochs)

In [ ]:
model_m = YOLO("yolov8s.pt")

results_m = model_m.train(
    data=DATA_CONFIG,
    epochs=50,
    imgsz=640,
    batch=16,
    name="yolov8m_bdd100k_v1",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    patience=15,
    save=True,
    plots=True,
    **AUGMENTATION,
)

## Save Outputs

In [ ]:
best_weights_src = os.path.join(PROJECT_DIR, "train/yolov8m_bdd100k_v1/weights/best.pt")
results_csv_src = os.path.join(PROJECT_DIR, "train/yolov8m_bdd100k_v1/results.csv")

if os.path.exists(best_weights_src):
    shutil.copy(best_weights_src, "/kaggle/working/yolov8m_bdd100k_best.pt")
    print(f"Best weights saved to /kaggle/working/yolov8m_bdd100k_best.pt")

if os.path.exists(results_csv_src):
    shutil.copy(results_csv_src, "/kaggle/working/yolov8m_results.csv")
    print(f"Results CSV saved to /kaggle/working/yolov8m_results.csv")

run_dir = os.path.join(PROJECT_DIR, "train/yolov8m_bdd100k_v1")
if os.path.exists(run_dir):
    print(f"\nFull run directory: {run_dir}")
    for f in sorted(os.listdir(run_dir)):
        print(f"  {f}")

In [ ]:
import pandas as pd

results_csv = "/kaggle/working/yolov8m_results.csv"
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"Training completed: {len(df)} epochs")
    print(f"Best mAP50: {df['metrics/mAP50(B)'].max():.4f}")
    print(f"Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
    display(df.tail())

In [ ]:
import shutil
from IPython.display import FileLink

folder_path = "/kaggle/working/bdd100k_project"

zip_path = "/kaggle/working/bdd100k_project.zip"

shutil.make_archive(zip_path.replace('.zip',''), 'zip', folder_path)

# Display a download link
FileLink(zip_path)